In [1]:
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

from internal.data_types import HistologyDataset
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
train_loader = data.train_loader
val_loader = data.val_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
is_cuda_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
num_classes = 4

model = timm.create_model(
    # 'convnext_tiny',
    'tf_efficientnetv2_s.in21k',
    pretrained=True,
    num_classes=num_classes
)
model = model.to(device)

for name, param in model.named_parameters():
    if "head" not in name:   # convnext uses 'head' for classifier
        param.requires_grad = False

criterion = nn.CrossEntropyLoss()

head_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=10
)


/home/andre/university/AN2DL-Challenge-2/.venv/lib/python3.12/site-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnetv2_s_in21k to current tf_efficientnetv2_s.in21k.
  model = create_fn(


In [3]:
# class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)
# class_weights = (class_counts.sum() / class_counts)  # inverse frequency
# class_weights = class_weights / class_weights.mean() # normalize a bit
# class_weights = class_weights.to(device)
#
# criterion = nn.CrossEntropyLoss(weight=class_weights)

In [4]:
# optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer, T_max=20
# )

In [5]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for imgs, labels in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu())
        all_targets.append(labels.detach().cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro')

    print(f"[] t_loss={epoch_loss:.4f} | F1(macro)={f1:.4f} | Acc={acc:.4f}")

    return epoch_loss, acc, f1

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for imgs, labels in tqdm(loader, desc="Val", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(labels.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro')

    return epoch_loss, acc, f1


In [6]:
EPOCHS = 7
best_f1 = 0.0
best_state = None

for epoch in range(1, EPOCHS+1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc, val_f1 = validate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    print(
        f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
        f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, "best_freeze_convnext_tiny.pth")
        print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")



Epoch 1/7


[] t_loss=4.9026 | F1(macro)=0.2645 | Acc=0.2896


Train  loss=4.9026 acc=0.2896 f1=0.2645 | Val loss=6.6579 acc=0.2686 f1=0.2572
  🔥 New best F1: 0.2572 – model saved.

Epoch 2/7


[] t_loss=4.2314 | F1(macro)=0.3158 | Acc=0.3463


Train  loss=4.2314 acc=0.3463 f1=0.3158 | Val loss=7.5468 acc=0.2968 f1=0.2541

Epoch 3/7


...

Train  loss=2.7312 acc=0.4517 f1=0.4181 | Val loss=6.2124 acc=0.2898 f1=0.2394

Epoch 6/7


[] t_loss=2.5009 | F1(macro)=0.4220 | Acc=0.4517


Train  loss=2.5009 acc=0.4517 f1=0.4220 | Val loss=4.8028 acc=0.3216 f1=0.3047
  🔥 New best F1: 0.3047 – model saved.

Epoch 7/7


[] t_loss=2.2272 | F1(macro)=0.4426 | Acc=0.4659


Train  loss=2.2272 acc=0.4659 f1=0.4426 | Val loss=4.9107 acc=0.2898 f1=0.2401


In [7]:
# Unfreeze entire model
for param in model.parameters():
    param.requires_grad = True

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=15
)

# OPTIONAL: re-introduce mild class weights
class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)
class_weights = (class_counts.sum() / class_counts)
class_weights = class_weights / class_weights.mean()
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))


In [8]:
EPOCHS = 10
best_f1 = 0.0
best_state = None

for epoch in range(1, EPOCHS+1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc, val_f1 = validate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    print(
        f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
        f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, "best_freeze_convnext_tiny.pth")
        print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")



Epoch 1/10


[] t_loss=2.9584 | F1(macro)=0.3718 | Acc=0.3897


Train  loss=2.9584 acc=0.3897 f1=0.3718 | Val loss=4.2640 acc=0.2509 f1=0.2128
  🔥 New best F1: 0.2128 – model saved.

Epoch 2/10


[] t_loss=2.0850 | F1(macro)=0.3900 | Acc=0.4066


Train  loss=2.0850 acc=0.4066 f1=0.3900 | Val loss=2.9030 acc=0.2898 f1=0.2700
  🔥 New best F1: 0.2700 – model saved.

Epoch 3/10


...

Train  loss=0.9204 acc=0.6289 f1=0.6219 | Val loss=2.2561 acc=0.3569 f1=0.3187

Epoch 9/10


[] t_loss=0.7975 | F1(macro)=0.6718 | Acc=0.6740


Train  loss=0.7975 acc=0.6740 f1=0.6718 | Val loss=2.1780 acc=0.3286 f1=0.3163

Epoch 10/10


[] t_loss=0.7402 | F1(macro)=0.6894 | Acc=0.6838


Train  loss=0.7402 acc=0.6838 f1=0.6894 | Val loss=2.2843 acc=0.2968 f1=0.2719


In [11]:
test_dataset = HistologyDataset(
    data.test_df,
    transforms=data.val_test_transforms,  # same as validation
    is_train=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,        # or 32 if fits
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [12]:
# load best_state
model.load_state_dict(torch.load("best_freeze_convnext_tiny.pth", map_location=device))
model.to(device)
model.eval()

EfficientNet(
  (conv_stem): Conv2dSame(3, 24, kernel_size=(3, 3), stride=(2, 2), bias=False)
  (bn1): BatchNormAct2d(
    24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): ConvBnAct(
        (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNormAct2d(
          24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (drop_path): Identity()
      )
      (1): ConvBnAct(
        (conv): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNormAct2d(
          24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (drop_path):

In [13]:
all_sample_indices = []
all_pred_labels = []

with torch.no_grad():
    for imgs, sample_indices in test_loader:
        imgs = imgs.to(device, non_blocking=True)

        logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()  # [B]

        for si, p in zip(sample_indices, preds):
            all_sample_indices.append(si)
            all_pred_labels.append(data.idx2label[int(p)])


In [14]:
# Ensure ".png" in the name
sample_index_with_ext = [f"{si}.png" if not si.endswith(".png") else si
                         for si in all_sample_indices]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": all_pred_labels
})

submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

   sample_index            label
0  img_0000.png          HER2(+)
1  img_0001.png  Triple negative
2  img_0002.png        Luminal A
3  img_0003.png  Triple negative
4  img_0004.png          HER2(+)
